In [1]:
!pip install -q pypdf sentence-transformers google-generativeai gradio scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 5.4 MB/s eta 0:00:00


In [2]:
from google.colab import files, userdata
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import google.generativeai as genai
import gradio as gr
import numpy as np

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
uploaded = files.upload()

Saving 30X_Doc3_Equipo_Herramientas.pdf to 30X_Doc3_Equipo_Herramientas.pdf
Saving 30X_Doc2_Programas_Operacion.pdf to 30X_Doc2_Programas_Operacion.pdf
Saving 30X_Doc1_Organizacion.pdf to 30X_Doc1_Organizacion.pdf


In [4]:
pdfs = [
    "30X_Doc1_Organizacion.pdf",
    "30X_Doc2_Programas_Operacion.pdf",
    "30X_Doc3_Equipo_Herramientas.pdf"
]

texto_completo = ""

for pdf in pdfs:
    reader = PdfReader(pdf)
    for page in reader.pages:
        text = page.extract_text()
        if text:
            texto_completo += text + "\n"

print("Documentos cargados ✅")
print("Caracteres:", len(texto_completo))

Documentos cargados ✅
Caracteres: 10197


In [5]:
def crear_chunks(texto, chunk_size=900, overlap=150):
    chunks = []
    start = 0

    while start < len(texto):
        end = start + chunk_size
        chunks.append(texto[start:end])
        start += chunk_size - overlap

    return chunks

chunks = crear_chunks(texto_completo)

print("Fragmentos creados ✅")
print("Número de fragmentos:", len(chunks))
print("\nPrimer fragmento:\n")
print(chunks[0][:700])

Fragmentos creados ✅
Número de fragmentos: 14

Primer fragmento:

30X — Quiénes somos y cómo funcionamos
Documento de onboarding · Versión interna 2025
¿Qué es 30X?
30X es una escuela de negocios ejecutiva y plataforma de red para founders, CEOs y altos directivos de
empresas hispanohablantes que quieren dejar de sobrevivir y empezar a escalar. Combinamos inmersiones
presenciales de 3 días con programas online en vivo, casos reales, frameworks probados y una red de más
de 2.000 ejecutivos.
Fue fundada por Andrés Bilbao (cofundador de Rappi), Daniel Bilbao (cofundador de Truora) y Dylan
Rosemberg. La premisa central: los mejores profesores son quienes construyeron empresas reales, no
quienes enseñaron teoría.
Presencia actual
País
Estado
Formato
Colombia
Ac


In [6]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embedding_model.encode(chunks)

print("Embeddings creados ✅")
print("Shape:", chunk_embeddings.shape)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings creados ✅
Shape: (14, 384)


In [7]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def buscar_contexto(pregunta, top_k=3):

    pregunta_embedding = embedding_model.encode([pregunta])

    similitudes = cosine_similarity(
        pregunta_embedding,
        chunk_embeddings
    )[0]

    indices = np.argsort(similitudes)[::-1][:top_k]

    contexto = "\n\n---\n\n".join(
        [chunks[i] for i in indices]
    )

    return contexto

In [8]:
import google.generativeai as genai

genai.configure(api_key="APP_KEY")

model = genai.GenerativeModel("gemini-2.5-flash")

print("Gemini conectado ✅")

Gemini conectado ✅


In [ ]:


historial = []

def responder_30x(pregunta):

    contexto = buscar_contexto(pregunta, top_k=1)

    historial_texto = "\n".join(historial[-4:])

    prompt = f"""
Eres el asistente oficial de onboarding de 30X.

REGLAS:
- Usa únicamente la información del contexto.
- No inventes información.
- Resume y organiza la respuesta.
- Utiliza listas y subtítulos cuando sea útil.
- Responde como si estuvieras ayudando a una persona nueva en la empresa.
- Si la respuesta no está en el contexto, responde exactamente:

"No encontré esa información en los documentos de onboarding. Te recomiendo consultarlo con el Chief of Staff."

CONTEXTO:
{contexto}

PREGUNTA:
{pregunta}

RESPUESTA:
"""

    try:
        respuesta = model.generate_content(prompt).text

        historial.append(f"Usuario: {pregunta}")
        historial.append(f"Agente: {respuesta}")

        return respuesta

    except Exception as e:
        return f"⚠️ Error generando respuesta: {str(e)}"


# ==========================
# INTERFAZ GRADIO
# ==========================

custom_css = """
.gradio-container {
    max-width: 1000px !important;
    margin: auto !important;
}

footer {
    visibility: hidden;
}

h1 {
    text-align: center;
}
"""


def chat_30x(message, history):
    return responder_30x(message)


demo = gr.ChatInterface(
    fn=chat_30x,

    title="🚀 30X Onboarding Assistant",

    description="""
Tu asistente de onboarding para resolver dudas sobre cultura, operación, herramientas y programas de 30X.
""",

    examples=[
        "¿Qué es 30X y quiénes la fundaron?",
        "¿Qué herramientas usa el equipo?",
        "¿Cómo funciona una cohorte online?",
        "¿Qué se espera de mí durante la primera semana?",
        "¿A quién le escribo si tengo una pregunta que no aparece en los documentos?"
    ],

    theme=gr.themes.Soft(),
    css=custom_css
)

demo.launch(
    share=True,
    debug=True
)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e7f77bff1f4b8d3c1e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
